# セミナーII 第1回 実践（ダイジェスト版）：画像分類モデルを学習する

本編 `01_image_classification.ipynb` の要点だけを、短い時間で動かせるようにしたノートブックです。題材は本編と同じで、犬と猫の写真を37クラス（品種）に分類します（Oxford-IIIT Pet）。

プログラミングや機械学習に慣れていなくても、上のセルから順に実行すれば最後まで動きます。コードを読めなくてもかまいません。各セルの前の説明と、実行した結果を見ることを優先してください。

**このノートブックで分かること**

1. 学習に使うデータは「画像と正解の組」で、画像はコンピュータにとって数の並びであること
2. 学習は「予測を計算 → Loss を計算 → 勾配を計算 → $\theta$ を更新」の繰り返しで、繰り返すと Loss が下がり、正解率が上がること
3. 人間が決める学習の設定（学習済みの重みを使うか、学習率など）で、結果が大きく変わること

**最初に**（アカウントの用意や保存などの手順は、リポジトリの `lesson01/colab_guide.md` にあります）

1. メニューの「ファイル」→「ドライブにコピーを保存」で、自分のコピーを作ります。
2. 「ランタイム」→「ランタイムのタイプを変更」で **GPU**（T4 など）を選びます。
3. 上のセルから順に実行します。セルの左の ▶ を押すか、セルを選んで **Shift + Enter** を押すと実行できます。実行中は ▶ の周りが回り、終わるとセルの下に結果が出ます。

全体で約35分（うち計算 約5分）です。各節の見出しの下に目安を書いています。見出しの（p〇〇）は、対応するスライドのページです。

**うまくいかないとき**

- しばらく操作しないと接続が切れ、それまでの計算結果が消えます。1. から順に実行し直してください。
- `NameError`（名前が定義されていない）が出たら、前のセルを飛ばしていないか確かめ、1. から順に実行し直してください。
- それでも直らないときは、エラーの最後の数行と、どのセルで出たかを添えて質問してください。

Colab で GPU が使えないときは、Kaggle 版（`01_image_classification_digest_kaggle.ipynb`、手順は `kaggle_guide.md`）を使ってください。

## 0. 全体の流れ

目安：約2分

画像分類のモデルは、次の式で表せました（p7、p16）。

$$Y = F(X; \theta)$$

$X$ は入力の画像、$Y$ は答え（クラス）、$F$ はネットワークの形、$\theta$ はパラメータ（重み）です。学習とは、予測 $\hat{Y}$ と正解 $Y$ のずれ（Loss）が小さくなるように、$\theta$ を少しずつ動かすことでした。

| 節 | やること | 学習に用意するもの（p89） |
| --- | --- | --- |
| 1. 準備 | GPU を使えるようにし、学習の設定を決める | 計算環境、学習の設定 |
| 2. データを見る | 画像と正解の組を見る | データセット |
| 3. モデルを用意する | 学習済みの ResNet-18 を読み込み、学習の前に予測してみる | モデル $F$ |
| 4. 学習を1回だけ動かす | 学習の繰り返しを、4行のコードで1回だけ実行する | Loss、更新の方法 |
| 5. 繰り返して学習する | 4. の4行を何百回も繰り返す | |
| 6. 結果を確かめる | test の正解率、間違えた画像、学習の後の予測 | |
| 7. 自分の画像で試す | 自分で撮った画像を入れる | |
| 8. 設定を変えて比べる | 学習の設定を変えて学習し直し、結果を比べる | |

## 1. 準備（p102）

目安：約3分（うち計算 約1分）

教材のファイル（`.py` と画像）を取得し、使うライブラリを読み込みます。学習の手順のうち、長いものは `.py` のファイルに入れてあり、このノートブックでは名前を呼び出すだけにしています（中身は本編のセルに書いてあります）。

| ファイル | 中身 |
| --- | --- |
| `data.py` | データの読み込み、前処理、mini-batch |
| `models.py` | モデル $F$ を作る |
| `engine.py` | 学習、評価、推論 |
| `viz.py` | 図を表示する（学習の仕組みとは関係しない） |

実行すると、最後に `device: cuda` と GPU の名前が表示されます。`device: cpu` と表示されたら GPU が選ばれていないので、「ランタイム」→「ランタイムのタイプを変更」で GPU を選んで、もう一度実行してください。

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/r-terasawa457/seminar2-2026.git"
REPO_REF = "main"  # 授業当日は、教材が変わらないようタグを指定する

if not Path("viz.py").exists():  # Colab では、最初はファイルが無い
    if not Path("seminar2-2026").exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, "seminar2-2026"],
                       check=True)
    os.chdir("seminar2-2026/lesson01")
if os.getcwd() not in sys.path:  # import で .py を見つけられるようにする
    sys.path.insert(0, os.getcwd())

# Colab に入っていないライブラリがあれば入れる
import importlib.util

for package in ["timm", "datasets"]:
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import numpy as np
import torch
import torch.nn as nn
from PIL import Image

import data    # データを用意する（data.py）
import models  # モデル F を作る（models.py）
import engine  # 学習、評価、推論（engine.py）
import viz     # 図を表示する（viz.py）

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "（GPU を選ぶと速くなります）")

### 学習の設定（p100）

$\theta$ は機械が求めますが、次の値（学習の設定）は人間が決めます。8. で、このうち1つを変えて比べます。

In [ ]:
MODEL_NAME = "resnet18.a1_in1k"  # モデル F（timm の名前）
PRETRAINED = True                # 学習済みの重みから始めるか（p101）
EPOCHS = 3                       # データ全体を何周するか（p98）
BATCH_SIZE = 64                  # 1回の更新に使う枚数（p99）
LEARNING_RATE = 1e-3             # 学習率：θ を1回に動かす幅を決める（p27）
SEED = 0                         # 乱数の種。同じなら、データの分け方が同じになる
torch.manual_seed(SEED)

# GPU が無いときは、枚数を減らして早く終わらせる（None は全部）
MAX_TRAIN_IMAGES = None if device == "cuda" else 600
MAX_TEST_IMAGES = None if device == "cuda" else 300
NUM_WORKERS = 2 if device == "cuda" and os.name != "nt" else 0  # データを読む並列数

## 2. データを見る（p91〜p93）

目安：約5分（うち初回のダウンロード 約1分）

Oxford-IIIT Pet は、犬と猫の写真に、品種名の正解ラベルを付けたデータセットです。Hugging Face Hub から取得します（約800MB、初回のみ）。ログインは不要で、`HF_TOKEN` に関する警告が表示されても無視してかまいません。

取得したデータは、役割の違う3つに分けます（p93）。学習に使っていないデータで評価するためです。

| データ | 役割 | たとえ |
| --- | --- | --- |
| train | $\theta$ の更新に使う | 練習問題 |
| validation | 学習の途中で成績を測り、一番良かった時点の $\theta$ を選ぶ | 模試 |
| test | 最後に1回だけ測る | 本番の試験 |

画像の著作権は元の撮影者にあり、データセットは CC BY-SA 4.0 で公開されています（O. M. Parkhi, A. Vedaldi, A. Zisserman, C. V. Jawahar, "Cats and Dogs", CVPR 2012）。

In [ ]:
train_data, val_data, test_data, class_names = data.load_splits(
    seed=SEED, max_train=MAX_TRAIN_IMAGES, max_test=MAX_TEST_IMAGES)
num_classes = len(class_names)
print("train      :", len(train_data), "枚")
print("validation :", len(val_data), "枚")
print("test       :", len(test_data), "枚")
print("クラスの数 :", num_classes)

# 8クラスの見本。括弧の中はクラスの番号
viz.show_samples(train_data, class_names,
                 ["Persian", "Siamese", "Sphynx", "British_Shorthair", "shiba_inu", "pug", "beagle", "samoyed"])

### データ1件は、画像と正解の組（p92、p47）

データの1件は、入力の画像 $X$ と、正解のラベル $Y$（クラスの番号）の組です。クラス名は 0〜36 の番号で扱います。

コンピュータにとって、画像は「高さ × 幅 × 3（赤、緑、青）」個の、0〜255 の整数の並びです。モデルが受け取るのは、この数の並びです。

In [ ]:
# スライドと同じ柴犬の写真（見つからないときは先頭の1件）
ids = train_data["image_id"]
sample = train_data[ids.index("shiba_inu_178")] if "shiba_inu_178" in ids else train_data[0]
print("正解ラベル Y:", sample["label"], "→", class_names[sample["label"]])
viz.show_example(sample, class_names)

pixels = np.array(sample["image"].convert("RGB"))
h, w = pixels.shape[:2]
print("画像 X の形 (高さ, 幅, 色):", pixels.shape)
print("中央の 4×4 画素の、赤の値:")
print(pixels[h // 2:h // 2 + 4, w // 2:w // 2 + 4, 0])

## 3. モデルを用意する（p79、p101）

目安：約3分

CNN の一つ ResNet-18 を使います。画像の特徴を取り出す**特徴抽出器**と、特徴からクラスのスコアを出す**分類ヘッド**からできています（p79）。

特徴抽出器には、別の大きなデータセット ImageNet（約128万枚、1000クラス）で学習済みの重みを読み込みます。分類ヘッドは37クラス用に付け替えるので、その重みはランダムな値から始まります。一から学習するより、少ないデータと短い時間で済みます（転移学習。p101）。

In [ ]:
model = models.build_model(MODEL_NAME, num_classes, pretrained=PRETRAINED).to(device)

# θ は1つの数ではなく、たくさんのパラメータ（カーネルの値、全結合層の重みなど）の集まり
n_params = sum(p.numel() for p in model.parameters())
print(f"θ に含まれるパラメータの数: {n_params:,} 個")

### 学習の前に予測してみる（p83、p94）

猫の写真を入れてみます。モデルに入れる前に、画像の大きさ（224×224）と値の範囲をそろえる**前処理**をします（p94）。

モデルは37クラスそれぞれのスコアを出し、それを softmax で合計1の確率に変換します（p83）。分類ヘッドはまだ学習していないので、ほとんど当てずっぽうで、どのクラスも確率が低くなります。6. で、学習した後のモデルと比べます。

In [ ]:
train_transform, eval_transform = data.build_transforms()  # 前処理。学習用はデータ拡張つき（p94、p95）

# 猫の写真（scikit-image の data.chelsea()。撮影 Stefan van der Walt、CC0）
cat = Image.open("assets/chelsea.png")
probs = engine.predict(model, cat, eval_transform, device)  # 37クラスの確率
print(f"一番高い確率: {probs.max().item():.1%}  （当てずっぽうなら 1/{num_classes} ≒ {1 / num_classes:.1%}）")
viz.show_prediction(cat, probs, class_names)

## 4. 学習を1回だけ動かす（p85、p86）

目安：約5分

学習に用意するもののうち、残りの2つを決めます。

- **Loss**：分類では交差エントロピーを使います。正解のクラスの確率を $p$ とすると、Loss ＝ $-\log p$ です。正解の確率が1に近いほど 0 に近く、0 に近いほど大きくなります（p85）。
- **更新の方法**：勾配降下法で、Loss が小さくなる方向に $\theta$ を少し動かします（p26）。ここでは、その改良版の AdamW を使います。

画像は1枚ずつではなく、64枚まとめて計算します（**mini-batch**。p99）。`DataLoader` が、前処理をした画像を64枚ずつ束ねて取り出します。学習用の画像には、毎回ランダムに切り抜きと左右反転（データ拡張。p95）をかけ、同じ画像を丸覚えしにくくします。

In [ ]:
train_loader = data.make_loader(train_data, train_transform, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = data.make_loader(val_data, eval_transform, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = data.make_loader(test_data, eval_transform, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

criterion = nn.CrossEntropyLoss()                                    # Loss：交差エントロピー
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)  # 更新の方法：θ の全部を動かす

batch = next(iter(train_loader))  # mini-batch を1つ取り出す
print("入力 X の形 (枚数, 色, 高さ, 幅):", tuple(batch["image"].shape))
print("正解 Y の形 (枚数,):", tuple(batch["label"].shape))
viz.show_batch(batch, class_names)

取り出した mini-batch で、学習の繰り返しを**1回だけ**実行します。スライドの4つの手順が、コードの4行に対応しています（p86）。

| 学習の繰り返し | コード |
| --- | --- |
| ① 予測を計算する $\hat{Y} = F(X; \theta)$ | `logits = model(images)` |
| ② Loss を計算する | `loss = criterion(logits, labels)` |
| ③ 勾配を計算する（誤差逆伝播） | `loss.backward()` |
| ④ $\theta$ を更新する | `optimizer.step()` |

同じ64枚で測った Loss が、更新の後に下がれば成功です。

In [ ]:
images = batch["image"].to(device)  # 入力 X（64枚の画像）
labels = batch["label"].to(device)  # 正解 Y（64個のクラスの番号）

model.train()
logits = model(images)            # ① 予測を計算する：Ŷ = F(X; θ)
loss = criterion(logits, labels)  # ② Loss を計算する
optimizer.zero_grad()             #    （前回の勾配を消す）
loss.backward()                   # ③ 勾配を計算する（誤差逆伝播）
optimizer.step()                  # ④ θ を更新する

with torch.no_grad():  # 同じ64枚で、もう一度 Loss を測る（θ は動かさない）
    after = criterion(model(images), labels).item()
print(f"同じ mini-batch の Loss: {loss.item():.3f}（更新前） → {after:.3f}（更新後）")

## 5. 繰り返して学習する（p97〜p99）

目安：約3分（うち計算 約1.5分）

4. の4行を、mini-batch を替えながら繰り返します。train の全部を1周すると **1 epoch** です。これを `EPOCHS` 回（3周）繰り返し、1周ごとに validation で正解率を測って、一番良かった時点の $\theta$ を残します（p97）。

`engine.fit` がこの繰り返しをします。中では、4. と同じ4行が動いています（`engine.py` の `train_one_epoch`）。中身を見たいときは、セルの最後の行の先頭の `#` を消して実行してください。

1周ごとに1行ずつ、train と validation の Loss（`loss`）と正解率（`acc`）が表示されます。

In [ ]:
print(f"1 epoch で θ を更新する回数: {len(train_loader)} 回、{EPOCHS} epoch で {EPOCHS * len(train_loader)} 回")
history, best_acc = engine.fit(model, train_loader, val_loader, criterion, optimizer, device, EPOCHS)

# import inspect; print(inspect.getsource(engine.train_one_epoch))  # 中身を表示する

### 学習曲線（p104）

epoch ごとの Loss（左）と正解率（右）です。train の Loss が下がっていれば、学習が進んでいます。validation の Loss が途中から上がり始めたら、train を丸覚えし始めたサイン（過学習）です。

In [ ]:
viz.plot_history(history)

## 6. 結果を確かめる（p105、p17、p106）

目安：約5分

### test で測る

学習にも、$\theta$ を選ぶのにも使っていない test で、最後に1回だけ正解率を測ります。37クラスなので、当てずっぽうなら約3%です。同じ設定でも、正解率は実行のたびに数ポイント変わります。

下の図は、間違いが多かった組み合わせから1枚ずつ、間違えた画像を並べたものです。右の2列は、正解のクラスと予測したクラスの見本（train から1枚ずつ）です。見比べて、なぜ間違えたのかを考えてみましょう。

In [ ]:
test_loss, test_acc, preds, test_labels = engine.evaluate(model, test_loader, criterion, device)
print(f"validation acc {best_acc:.3f}  （一番良かった時点）")
print(f"test       acc {test_acc:.3f}  （{len(test_labels)} 枚）")
print(f"参考：当てずっぽうなら 1/{num_classes} ≒ {1 / num_classes:.3f}")

viz.show_mistakes(test_data, preds, test_labels, class_names, reference=train_data)

### 学習の後に、もう一度予測する（p17）

3. で試した猫の写真を、学習した後のモデルに入れます。学習が終われば、画像を入れると答えが出ます（$\theta$ は学習で求めた値のまま）。下の段は、上位のクラスの見本です。3. の結果と比べてみましょう。

In [ ]:
probs = engine.predict(model, cat, eval_transform, device)
viz.show_prediction(cat, probs, class_names, reference=train_data)

### 学習していないものを入れると（p106）

コーヒーカップの写真を入れます。モデルは、学習した37クラスのどれかでしか答えられません。確率は合計が1になるように変換しただけなので、高くても正しいとは限りません。

In [ ]:
# コーヒーカップの写真（scikit-image の data.coffee()。撮影 Rachel Michetti、CC0）
coffee = Image.open("assets/coffee.png")
probs = engine.predict(model, coffee, eval_transform, device)
viz.show_prediction(coffee, probs, class_names, reference=train_data)

## 7. 自分の画像で試す

目安：約3分（画像を用意する時間は除く）

自分で撮った画像を入れてみましょう。セルの右に出る `USE_UPLOAD` にチェックを入れて（または `False` を `True` に書き換えて）実行すると、アップロードのボタンが表示されます。犬や猫以外（タヌキ、ぬいぐるみ、自動車など）も試してください。

In [ ]:
USE_UPLOAD = False  # @param {type:"boolean"}

if USE_UPLOAD:
    from google.colab import files

    for name in files.upload():
        image = Image.open(name)
        viz.show_prediction(image, engine.predict(model, image, eval_transform, device), class_names,
                            reference=train_data)

## 8. 設定を変えて比べる（p100）

目安：約5分（うち計算 1回あたり約1.5分。選択肢によっては約3分）

学習の設定を変えて、3.〜6. と同じ手順で学習し直し、5. の結果と比べます。次の表から選びます。**実行する前に、「確かめること」への答えを予想してください。** 時間があれば、2〜3個試して並べてみましょう。

| 分類 | 選択肢 | 変える設定 | 確かめること | 計算の時間 |
| --- | --- | --- | --- | --- |
| データ | データ拡張を使わない | `augment=False` | データ拡張（ランダムな切り抜きと左右反転）は、どのくらい役に立っているか（p95） | 約1.5分 |
| データ | batch サイズを16にする | `batch_size=16` | 1回の更新に使う枚数を減らす（更新の回数は4倍になる）と、どうなるか（p99） | 約2分 |
| モデル | 学習済みの重みを使わない | `pretrained=False` | ImageNet で学習した重みは、どのくらい役に立っているか（p101） | 約1.5分 |
| モデル | 学習済みの重みを使わずに、epoch を8にする | `pretrained=False, epochs=8` | 長く学習すれば、学習済みの重みが無くても追いつくか | 約3分 |
| モデル | 大きいモデル（ResNet-50）にする | `model="resnet50.a1_in1k"` | パラメータが約2倍のモデルにすると、どうなるか | 約3分 |
| 更新の方法 | 学習率を100倍にする | `lr=0.1` | $	heta$ を1回に大きく動かしすぎると、Loss はどうなるか（p27） | 約1.5分 |
| 更新の方法 | 学習率を100分の1にする | `lr=1e-05` | $	heta$ を少しずつしか動かさないと、どうなるか（p27） | 約1.5分 |
| 学習の長さ | epoch を1にする | `epochs=1` | 1周だけで、どこまで学習できるか（p98） | 約0.5分 |
| 学習の長さ | epoch を8にする | `epochs=8` | 長く学習すると、validation の Loss はどうなるか（過学習。p104） | 約3分 |
| | 自分で決める | `CUSTOM` に書いた設定 | 上の組み合わせや、別の値を自分で考えて試す | |

まず、5. の結果を比べる基準（表の #0）として入れておきます。このセルは1回だけ実行してください（もう一度実行すると、表が最初に戻ります）。

In [ ]:
# 5. の学習の設定（1. で決めたもの。augment はデータ拡張を使うか）
BASE_SETTINGS = {"model": MODEL_NAME, "pretrained": PRETRAINED, "augment": True,
                 "batch_size": BATCH_SIZE, "lr": LEARNING_RATE, "epochs": EPOCHS}

# 5. の結果を、比べる基準（#0）として入れておく
results = [{"note": "5. の学習", "changes": "baseline", **BASE_SETTINGS,
            "best_val_acc": round(best_acc, 3), "test_acc": round(test_acc, 3), "history": history}]

次のセルで、`EXPERIMENT` に選択肢を選んで実行します。Colab では、セルの右に出るプルダウンから選べます（コードの `EXPERIMENT = "…"` の文字を書き換えても同じです）。別の選択肢を選んでもう一度実行すると、結果が表に加わります。5. で学習したモデルは上書きしません。

「自分で決める」を選んだときは、`CUSTOM` に、変えたい設定だけを書きます。書ける名前は、表の「変える設定」と同じ `model`、`pretrained`、`augment`、`batch_size`、`lr`、`epochs` です（例：`CUSTOM = {"lr": 3e-4, "epochs": 5}`）。

In [ ]:
EXPERIMENT = "学習済みの重みを使わない"  # @param ["データ拡張を使わない", "batch サイズを16にする", "学習済みの重みを使わない", "学習済みの重みを使わずに、epoch を8にする", "大きいモデル（ResNet-50）にする", "学習率を100倍にする", "学習率を100分の1にする", "epoch を1にする", "epoch を8にする", "自分で決める"]
CUSTOM = {"lr": 3e-4, "epochs": 5}  # 「自分で決める」のときに使う。変えたい設定だけを書く

changes = {
    "データ拡張を使わない": {"augment": False},
    "batch サイズを16にする": {"batch_size": 16},
    "学習済みの重みを使わない": {"pretrained": False},
    "学習済みの重みを使わずに、epoch を8にする": {"pretrained": False, "epochs": 8},
    "大きいモデル（ResNet-50）にする": {"model": "resnet50.a1_in1k"},
    "学習率を100倍にする": {"lr": 0.1},
    "学習率を100分の1にする": {"lr": 1e-5},
    "epoch を1にする": {"epochs": 1},
    "epoch を8にする": {"epochs": 8},
    "自分で決める": CUSTOM,
}[EXPERIMENT]
unknown = set(changes) - set(BASE_SETTINGS)
if unknown:
    raise ValueError(f"設定の名前が違います: {unknown}。書ける名前は {list(BASE_SETTINGS)} です")
settings = {**BASE_SETTINGS, **changes}  # 5. の設定のうち、選んだものだけを変える
print("今回の設定:", settings)

new_model = new_optimizer = None  # 前の実験のモデルを消して、GPU のメモリを空ける
torch.cuda.empty_cache()
torch.manual_seed(SEED)
new_train_loader = data.make_loader(train_data, train_transform if settings["augment"] else eval_transform,
                                    settings["batch_size"], shuffle=True, num_workers=NUM_WORKERS)  # 4.
new_model = models.build_model(settings["model"], num_classes, pretrained=settings["pretrained"]).to(device)  # 3.
new_optimizer = torch.optim.AdamW(new_model.parameters(), lr=settings["lr"])                                 # 4.
new_history, new_best_acc = engine.fit(new_model, new_train_loader, val_loader, criterion, new_optimizer,
                                       device, settings["epochs"])                                            # 5.
_, new_test_acc, _, _ = engine.evaluate(new_model, test_loader, criterion, device)                             # 6.
print(f"test acc {new_test_acc:.3f}")

results.append({"note": EXPERIMENT, "changes": ", ".join(f"{k}={v}" for k, v in changes.items()),
                **settings, "best_val_acc": round(new_best_acc, 3), "test_acc": round(new_test_acc, 3),
                "history": new_history})

結果を表と図で比べます。表の `best_val_acc` は validation の一番良かった正解率、`test_acc` は test の正解率です。図の見出しの #番号 は表の番号と同じで、#0 から変えた設定を表示します。

予想と比べてどうでしたか。正解率だけでなく、学習曲線の形も比べてください。同じ設定でも正解率は数ポイント変わるので、それより小さい差は、設定の違いによるものとは言えません。

In [ ]:
viz.show_results(results)

## 9. まとめ

目安：約2分

| 分かったこと | 節 |
| --- | --- |
| 学習に使うデータは、画像と正解の組。画像は数の並び | 2. |
| 学習済みのモデルから始めると、少ないデータと短い時間で学習できる | 3.、8. |
| 学習は「予測を計算 → Loss を計算 → 勾配を計算 → $\theta$ を更新」の繰り返し | 4.、5. |
| 学習の結果は、学習に使っていない test で確かめる | 6. |
| モデルは、学習したクラスのどれかでしか答えられない | 6.、7. |
| 学習の設定は人間が決め、結果を大きく左右する | 8. |

**課題**（p109）：課題の内容と確認項目は、本編の 10. にあります。このノートブックの結果（6. の間違えた画像、7. の自分の画像、8. の実験）を使って書いてかまいません。

**もっと知りたいとき**：本編 `01_image_classification.ipynb` では、同じ学習を節に分けて、関数の中身もセルに書いています。次のことも確かめられます。

| 内容 | 本編の節 |
| --- | --- |
| 前処理とデータ拡張の前と後の画像 | 3-3 |
| ResNet-18 の層を通るごとの形、ImageNet で学習したカーネル64枚 | 4. |
| 交差エントロピーが $-\log p$ と一致すること | 5-1 |
| 層ごとの特徴マップ | 8-1 |
| 実験の関数を使って、いくつもの設定を続けて試す。前処理のコードを直して比べる | 9. |
| 同じ学習を GitHub でよく見る形（`train.py`）で動かす | 11. |